# Notebook 7 / R7-K5 - WasteNet-256K Direct KD

Port K6 R7 ke protokol 5 kelas (TrashNet-K5, kelas `trash` dihapus). Melatih WasteNet-256K dengan logits-based direct knowledge distillation dari EfficientNet-B4 teacher R1-K5 pada seed yang sama.
R7-K5 menjawab apakah big-teacher direct KD membantu tiny student dibanding R6-K5 WasteNet-256K CE baseline (no-KD floor).

## Rules

- Gunakan `split_manifest_seed_*.csv` dari R0-K5 dan wajib verifikasi `dataset_name=TrashNet-K5`, `dataset_version=2`, `num_classes=5`, `trash` absen.
- Teacher EfficientNet-B4 harus berasal dari R1-K5 final pada seed yang sama.
- Pilot R7-K5 hanya memakai validation set (seed 42) untuk memilih `temperature` dan `alpha` via grid.
- Independent test set hanya dievaluasi ketika `RUN_PHASE = "final"` setelah winner dibekukan.
- Hyperparameter non-KD (IMG_SIZE=160, LR=0.03, 100 epoch) dibekukan identik dengan K6 R7 / R6-K5 agar 5-vs-6 tetap comparable.
- Final R7-K5 dijalankan 5 seed dengan konfigurasi yang sudah dibekukan.

## R7-K5 Pilot Sweep

Pilot menjalankan **3 config KD dalam satu Run All** (driver cell di bawah), seed 42 saja: `t2_a0p1`, `t2_a0p3`, `t4_a0p1` (alpha-row di T=2 + 1 cek T=4 buat re-verify T=2>T=4 di student 256K). Tiap config train student fresh + reproducible (init & urutan data identik, beda cuma T/alpha), simpan artefak ke folder sendiri di `pilots/R7/<setup_id>/`, lalu print + simpan `r7_k5_pilot_sweep_summary.csv` (sorted val acc).

Edit `PILOT_SWEEP` di Config kalau mau ganti/tambah config. Setelah lihat winner, set `FINAL_KD_TEMPERATURE`/`FINAL_KD_ALPHA`, `RUN_PHASE='final'`, jalankan 5 seed.

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import copy
import json
import os
import random
import time
import warnings

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=UserWarning)

print("Imports ready")


In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

class Config:
    EXPERIMENT_ID = "R7"
    EXPERIMENT_NAME = "R7-K5 WasteNet-256K Direct KD"
    PLATFORM = "Kaggle Notebooks"
    FRAMEWORK = "PyTorch"
    USE_AMP = True
    MULTI_GPU = False

    # Non-KD hyperparameters frozen identical to K6 R7 / R6-K5 so 5-vs-6 stays comparable.
    SEED = 3407  # Final seeds: 42, 123, 777, 2026, 3407
    RUN_PHASE = "final"  # "pilot" = 3-config KD sweep on seed 42; "final" = frozen 5-seed reporting

    DATASET_NAME = "TrashNet-K5"
    DATASET_VERSION = 2
    DATASET_DIR = Path(os.environ.get(
        "TRASHNET_K5_DATASET_DIR",
        "/kaggle/input/datasets/kholiqbudiman/trashnet-k5-waste-classification",
    ))

    R0_DIR_CANDIDATES = [
        Path(os.environ.get("R0_K5_DATA_PROTOCOL_DIR", "")),
        Path("/kaggle/working/final_research_kd_k5/r0_data_protocol"),
        Path("/kaggle/input/notebook0-r0-k5-data-protocol-setup/final_research_kd_k5/r0_data_protocol"),
        Path("/kaggle/input/notebook0-r0-k5-data-protocol-setup/r0_data_protocol"),
        Path("/kaggle/input/notebooks/hamzapratama/notebook0-r0-k5-data-protocol-setup/final_research_kd_k5/r0_data_protocol"),
        Path.cwd() / "final_research_kd_k5" / "r0_data_protocol",
    ]
    R1_TEACHER_ROOT_CANDIDATES = [
        Path(os.environ.get("R1_K5_TEACHER_DIR", "")),
        Path("/kaggle/working/final_research_kd_k5/runs/final/R1"),
        Path("/kaggle/input/output-notebook1-r1-k5-efficientnet-b4-teacher/final_research_kd_k5/runs/final/R1"),
        Path("/kaggle/input/output-notebook1-r1-k5-efficientnet-b4-teacher/runs/final/R1"),
        Path("/kaggle/input/notebook1-r1-k5-efficientnet-b4-teacher/final_research_kd_k5/runs/final/R1"),
        Path("/kaggle/input/notebook1-r1-k5-efficientnet-b4-teacher/runs/final/R1"),
        Path("/kaggle/input/notebooks/hamzapratama/notebook1-r1-k5-efficientnet-b4-teacher/final_research_kd_k5/runs/final/R1"),
        Path("/kaggle/input/r1-k5-final/final_research_kd_k5/runs/final/R1"),
        Path.cwd() / "final_research_kd_k5" / "runs" / "final" / "R1",
    ]

    DEFAULT_OUTPUT_ROOT = (
        Path("/kaggle/working/final_research_kd_k5/runs")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "final_research_kd_k5" / "runs"
    )
    OUTPUT_ROOT = Path(os.environ.get("R7_K5_OUTPUT_ROOT", str(DEFAULT_OUTPUT_ROOT)))

    MODEL_NAME = "WasteNet-256K"
    VARIANT_ID = "wastenet_256k"
    TRAINING_MODE = "direct_kd"
    TEACHER_MODEL_NAME = "EfficientNet-B4"
    TEACHER_TIMM_NAME = "efficientnet_b4"

    IMG_SIZE = 160  # WasteNet-256K design size (matches K6 R7 / R6-K5); do NOT change to 380.
    FINAL_EPOCHS = 100
    PILOT_EPOCHS = 60
    EPOCHS = PILOT_EPOCHS if RUN_PHASE == "pilot" else FINAL_EPOCHS
    BATCH_SIZE = 16
    NUM_WORKERS = 2

    LR = 0.03  # K6 R7 LR for the 256K family (matches R6-K5; bigger model than 128K).
    MOMENTUM = 0.9
    WEIGHT_DECAY = 1e-4
    SCHEDULER = "CosineAnnealingLR"

    # --- KD grid lookup (full K6 grid kept for reference / config provenance) ---
    PILOT_GRID_TEMPERATURES = [2.0, 4.0, 6.0]
    PILOT_GRID_ALPHAS = [0.05, 0.1, 0.3]
    PILOT_ANCHOR = {"temperature": 4.0, "alpha": 0.5}
    PILOT_CONFIGS = {
        "anchor_t4_a0p5": {"temperature": 4.0, "alpha": 0.5, "priority": 0},
        "t4_a0p1": {"temperature": 4.0, "alpha": 0.1, "priority": 1},
        "t2_a0p1": {"temperature": 2.0, "alpha": 0.1, "priority": 2},
        "t6_a0p1": {"temperature": 6.0, "alpha": 0.1, "priority": 3},
        "t4_a0p05": {"temperature": 4.0, "alpha": 0.05, "priority": 4},
        "t4_a0p3": {"temperature": 4.0, "alpha": 0.3, "priority": 5},
        "t2_a0p05": {"temperature": 2.0, "alpha": 0.05, "priority": 6},
        "t2_a0p3": {"temperature": 2.0, "alpha": 0.3, "priority": 7},
        "t6_a0p05": {"temperature": 6.0, "alpha": 0.05, "priority": 8},
        "t6_a0p3": {"temperature": 6.0, "alpha": 0.3, "priority": 9},
        "t2_a0p5": {"temperature": 2.0, "alpha": 0.5, "priority": 10},
    }
    # Main pilot sweep that picked the winner (seed 42): t2_a0p3 won; t2_a0p5 robustness check confirmed alpha>0.3 hurts.
    PILOT_SWEEP = ["t2_a0p1", "t2_a0p3", "t4_a0p1"]
    PILOT_PRESET = PILOT_SWEEP[0]  # representative; the sweep overrides per config

    # --- Final winner (FROZEN from K5 pilot sweep). Must be set before any final run. ---
    # Do NOT inherit the K6 winner (t2/a0.3) blindly; pick the K5 sweep winner on val.
    FINAL_KD_TEMPERATURE = 2.0  # FROZEN: K5 R7 pilot winner t2_a0p3 (best-among-KD; T=2 beats T=4)
    FINAL_KD_ALPHA = 0.3         # FROZEN: K5 R7 pilot winner t2_a0p3 (alpha peaks at 0.3; 0.5 hurt)

    if RUN_PHASE == "pilot":
        KD_TEMPERATURE = PILOT_CONFIGS[PILOT_PRESET]["temperature"]
        KD_ALPHA = PILOT_CONFIGS[PILOT_PRESET]["alpha"]
    else:
        KD_TEMPERATURE = FINAL_KD_TEMPERATURE
        KD_ALPHA = FINAL_KD_ALPHA

    CHECKPOINT_METRIC = "best_val_accuracy"
    EARLY_STOPPING = RUN_PHASE == "pilot"
    EARLY_STOPPING_MONITOR = "val_loss"
    PATIENCE = 15
    EVALUATE_TEST = RUN_PHASE == "final"


def float_tag(value):
    return str(value).replace(".", "p")


cfg = Config()


def build_pilot_setup_id(preset, temperature, alpha):
    return (
        f"r7_k5_pilot_wn256_direct-kd_{preset}_s{cfg.SEED}_"
        f"t{float_tag(temperature)}_a{float_tag(alpha)}_"
        f"kd{cfg.PILOT_EPOCHS}_"
        f"es-{cfg.EARLY_STOPPING_MONITOR.replace('_', '')}-p{cfg.PATIENCE}_"
        f"img{cfg.IMG_SIZE}_lr{float_tag(cfg.LR)}_bs{cfg.BATCH_SIZE}"
    )


VALID_FINAL_SEEDS = {42, 123, 777, 2026, 3407}
if cfg.RUN_PHASE not in {"pilot", "final"}:
    raise ValueError(f"Unsupported RUN_PHASE: {cfg.RUN_PHASE}")
if cfg.IMG_SIZE != 160:
    raise ValueError("R7-K5 must keep IMG_SIZE=160 to match K6 R7 WasteNet-256K.")
if cfg.RUN_PHASE == "pilot":
    if cfg.SEED != 42:
        raise ValueError("R7-K5 pilot sweep uses seed 42 only.")
    if not cfg.PILOT_SWEEP:
        raise ValueError("PILOT_SWEEP is empty.")
    for _preset in cfg.PILOT_SWEEP:
        if _preset not in cfg.PILOT_CONFIGS:
            raise ValueError(f"Unknown preset in PILOT_SWEEP: {_preset}. Choose from {list(cfg.PILOT_CONFIGS)}")
if cfg.RUN_PHASE == "final":
    if cfg.SEED not in VALID_FINAL_SEEDS:
        raise ValueError(f"Final seed must be one of {sorted(VALID_FINAL_SEEDS)}.")
    if cfg.EPOCHS != 100:
        raise ValueError("R7-K5 final protocol requires exactly 100 epochs.")
    if abs(cfg.LR - 0.03) > 1e-12:
        raise ValueError("R7-K5 final protocol freezes LR=0.03 (K6 R7 256K family).")
    if cfg.EARLY_STOPPING:
        raise ValueError("Early stopping must be disabled for final runs.")
    if cfg.KD_TEMPERATURE is None or cfg.KD_ALPHA is None:
        raise ValueError(
            "Set FINAL_KD_TEMPERATURE and FINAL_KD_ALPHA to the K5 R4 pilot winner before final runs. "
            "Do not inherit the K6 winner blindly."
        )
if cfg.EVALUATE_TEST != (cfg.RUN_PHASE == "final"):
    raise ValueError("Test evaluation must be disabled for pilot and enabled for final.")

if cfg.RUN_PHASE == "pilot":
    # Per-config SETUP_ID / OUTPUT_DIR are assigned inside the sweep driver.
    cfg.SETUP_ID = None
    cfg.OUTPUT_DIR = None
else:
    cfg.SETUP_ID = (
        f"r7_k5_final_wn256_direct-kd_s{cfg.SEED}_"
        f"t{float_tag(cfg.KD_TEMPERATURE)}_a{float_tag(cfg.KD_ALPHA)}_"
        f"kd{cfg.EPOCHS}_img{cfg.IMG_SIZE}_lr{float_tag(cfg.LR)}_bs{cfg.BATCH_SIZE}"
    )
    cfg.OUTPUT_DIR = cfg.OUTPUT_ROOT / "final" / cfg.EXPERIMENT_ID / f"seed_{cfg.SEED}"
    cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment : {cfg.EXPERIMENT_NAME}")
print(f"Run phase  : {cfg.RUN_PHASE}")
print(f"Seed       : {cfg.SEED}")
print(f"Epochs     : {cfg.EPOCHS}")
print(f"Img size   : {cfg.IMG_SIZE}")
print(f"Early stop : {cfg.EARLY_STOPPING} (patience={cfg.PATIENCE})")
print(f"Eval test  : {cfg.EVALUATE_TEST}")
print(f"Dataset dir: {cfg.DATASET_DIR}")
if cfg.RUN_PHASE == "pilot":
    print(f"Pilot sweep ({len(cfg.PILOT_SWEEP)} configs, one Run All):")
    for _p in cfg.PILOT_SWEEP:
        _c = cfg.PILOT_CONFIGS[_p]
        print(f"  - {_p}: T={_c['temperature']} alpha={_c['alpha']}")
else:
    print(f"Setup ID   : {cfg.SETUP_ID}")
    print(f"T / alpha  : {cfg.KD_TEMPERATURE} / {cfg.KD_ALPHA}")
    print(f"Output dir : {cfg.OUTPUT_DIR}")

In [ ]:
# ============================================================
# 3. Reproducibility and Device
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(cfg.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = cfg.USE_AMP and device.type == "cuda"
print(f"Device: {device}")
print(f"AMP enabled: {AMP_ENABLED}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
    print(f"VRAM GB: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")

## 1. Load R0-K5 Artifacts

Verifikasi `dataset_name=TrashNet-K5`, `dataset_version=2`, `num_classes=5`, `trash` absen, dan resolve teacher R1-K5 seed yang sama.

In [ ]:
# ============================================================
# 4. Resolve R0-K5 Artifacts and R1-K5 Teacher
# ============================================================

def has_r0_k5_artifacts(path: Path) -> bool:
    mapping_path = path / "class_mapping.json"
    if not mapping_path.exists() or not any(path.glob("split_manifest_seed_*.csv")):
        return False
    try:
        mapping = json.loads(mapping_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return (
        mapping.get("dataset_name") == cfg.DATASET_NAME
        and mapping.get("dataset_version") == cfg.DATASET_VERSION
        and mapping.get("num_classes") == 5
        and "trash" not in mapping.get("class_names", [])
    )


def candidate_variants(candidate: Path):
    yield candidate
    yield candidate / "r0_data_protocol"
    yield candidate / "final_research_kd_k5" / "r0_data_protocol"


def discover_r0_dirs():
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
    discovered = []
    for root in roots:
        if not root.exists():
            continue
        try:
            discovered.extend([path for path in root.rglob("r0_data_protocol") if path.is_dir()])
        except Exception as exc:
            print(f"Skipping R0 discovery under {root}: {exc}")
    return discovered


def resolve_existing_dir(candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if str(candidate) in {"", "."}:
            continue
        for variant in candidate_variants(candidate):
            if variant.exists() and has_r0_k5_artifacts(variant):
                return variant

    discovered = discover_r0_dirs()
    valid_discovered = [path for path in discovered if has_r0_k5_artifacts(path)]
    if valid_discovered:
        print("Auto-discovered R0 candidates:")
        for path in valid_discovered:
            print(f"- {path}")
        return valid_discovered[0]

    print("Checked R0 candidates:")
    for candidate in candidates:
        print(f"- {candidate}")
    print("Discovered r0_data_protocol dirs:")
    for path in discovered:
        print(f"- {path}")
    raise FileNotFoundError("R0-K5 data protocol directory not found. Set R0_K5_DATA_PROTOCOL_DIR.")


def resolve_teacher_checkpoint(seed: int) -> Path:
    filename = f"efficientnet_b4_teacher_r1_final_seed_{seed}_best.pth"
    candidates = []
    for root in cfg.R1_TEACHER_ROOT_CANDIDATES:
        root = Path(root)
        if str(root) in {"", "."}:
            continue
        candidates.extend([
            root / f"seed_{seed}" / filename,
            root / filename,
            root / "final_research_kd_k5" / "runs" / "final" / "R1" / f"seed_{seed}" / filename,
            root / "runs" / "final" / "R1" / f"seed_{seed}" / filename,
        ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    print("Checked teacher candidates:")
    for candidate in candidates:
        print(f"- {candidate}")
    raise FileNotFoundError(
        f"R1-K5 EfficientNet-B4 teacher checkpoint not found for seed {seed}. "
        "Attach R1-K5 final output or set R1_K5_TEACHER_DIR."
    )


R0_DIR = resolve_existing_dir(cfg.R0_DIR_CANDIDATES)
CLASS_MAPPING_PATH = R0_DIR / "class_mapping.json"
MANIFEST_PATH = R0_DIR / f"split_manifest_seed_{cfg.SEED}.csv"
EXCLUDED_DUPLICATES_PATH = R0_DIR / "excluded_duplicate_conflicts.csv"
TEACHER_CHECKPOINT_PATH = resolve_teacher_checkpoint(cfg.SEED)

for path in [CLASS_MAPPING_PATH, MANIFEST_PATH, EXCLUDED_DUPLICATES_PATH, TEACHER_CHECKPOINT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact missing: {path}")

with CLASS_MAPPING_PATH.open("r", encoding="utf-8") as handle:
    class_mapping = json.load(handle)

if class_mapping.get("dataset_name") != cfg.DATASET_NAME:
    raise ValueError(f"Wrong R0 dataset: {class_mapping.get('dataset_name')}")
if class_mapping.get("dataset_version") != cfg.DATASET_VERSION:
    raise ValueError(f"Wrong R0 dataset version: {class_mapping.get('dataset_version')}")
if class_mapping.get("num_classes") != 5 or "trash" in class_mapping.get("class_names", []):
    raise ValueError("R7-K5 requires exactly five non-trash classes.")

manifest_df = pd.read_csv(MANIFEST_PATH)
excluded_df = pd.read_csv(EXCLUDED_DUPLICATES_PATH)
if set(manifest_df["seed"].unique()) != {cfg.SEED}:
    raise ValueError(f"Manifest seed mismatch. Expected only {cfg.SEED}.")

CLASS_NAMES = class_mapping["class_names"]
CLASS_TO_IDX = class_mapping["class_to_idx"]
NUM_CLASSES = len(CLASS_NAMES)

print(f"R0 dir       : {R0_DIR}")
print(f"Manifest     : {MANIFEST_PATH.name}")
print(f"Teacher ckpt : {TEACHER_CHECKPOINT_PATH}")
print(f"Rows         : {len(manifest_df)}")
print(f"Classes      : {CLASS_NAMES}")
print(f"Excluded dup : {len(excluded_df)} rows")
display(manifest_df.groupby(["split", "label", "class_id"], as_index=False).size())

In [ ]:
# ============================================================
# 5. Manifest Validation
# ============================================================

required_columns = {"sample_id", "image_path", "relative_path", "label", "class_id", "split", "seed", "sha256"}
missing_columns = required_columns - set(manifest_df.columns)
if missing_columns:
    raise ValueError(f"Manifest missing columns: {sorted(missing_columns)}")

if set(manifest_df["seed"].unique()) != {cfg.SEED}:
    raise ValueError(f"Manifest seed mismatch. Expected only {cfg.SEED}.")

if set(manifest_df["split"].unique()) != {"train", "val", "test"}:
    raise ValueError("Manifest must contain train, val, and test splits.")

excluded_ids = set(excluded_df.get("sample_id", []))
if excluded_ids & set(manifest_df["sample_id"]):
    raise AssertionError("Excluded duplicate-conflict samples are present in this manifest.")

for class_name, class_id in CLASS_TO_IDX.items():
    rows = manifest_df[manifest_df["label"] == class_name]
    if rows.empty:
        raise AssertionError(f"Missing class in manifest: {class_name}")
    if set(rows["class_id"].unique()) != {class_id}:
        raise AssertionError(f"Class id mismatch for {class_name}")

print("Manifest validation passed")

## 2. Dataset and DataLoader

In [ ]:
# ============================================================
# 6. Transforms
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def coarse_dropout(img_size: int):
    min_h = int(img_size * 0.05)
    max_h = int(img_size * 0.20)
    try:
        return A.CoarseDropout(
            num_holes_range=(1, 1),
            hole_height_range=(min_h, max_h),
            hole_width_range=(min_h, max_h),
            fill=0,
            p=0.5,
        )
    except TypeError:
        return A.CoarseDropout(
            max_holes=1,
            min_height=min_h,
            max_height=max_h,
            min_width=min_h,
            max_width=max_h,
            fill_value=0,
            p=0.5,
        )


train_transform = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    coarse_dropout(cfg.IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print("Transforms ready")

In [ ]:
# ============================================================
# 7. Manifest Dataset
# ============================================================

class TrashNetManifestDataset(Dataset):
    def __init__(self, df: pd.DataFrame, dataset_dir: Path, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.dataset_dir = Path(dataset_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def resolve_path(self, row) -> Path:
        absolute_path = Path(row["image_path"])
        if absolute_path.exists():
            return absolute_path
        fallback_path = self.dataset_dir / row["relative_path"]
        if fallback_path.exists():
            return fallback_path
        raise FileNotFoundError(f"Image not found: {absolute_path} or {fallback_path}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.resolve_path(row)
        image = Image.open(image_path).convert("RGB")
        image = np.array(image)
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        return {
            "image": image,
            "label": int(row["class_id"]),
            "sample_id": row["sample_id"],
            "relative_path": row["relative_path"],
        }


train_df = manifest_df[manifest_df["split"] == "train"].copy()
val_df = manifest_df[manifest_df["split"] == "val"].copy()
test_df = manifest_df[manifest_df["split"] == "test"].copy()

train_dataset = TrashNetManifestDataset(train_df, cfg.DATASET_DIR, train_transform)
val_dataset = TrashNetManifestDataset(val_df, cfg.DATASET_DIR, eval_transform)
test_dataset = TrashNetManifestDataset(test_df, cfg.DATASET_DIR, eval_transform)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# ============================================================
# 8. DataLoaders and Distribution
# ============================================================

def worker_init_fn(worker_id):
    worker_seed = cfg.SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(cfg.SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
    generator=generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
)

split_counts = manifest_df.groupby(["split", "label", "class_id"], as_index=False).size()
display(split_counts)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

## 3. Student, Teacher, and Training Components

In [ ]:
# ============================================================
# 9. WasteNet-256K Model
# ============================================================

@dataclass(frozen=True)
class WasteNetVariant:
    variant_id: str
    display_name: str
    channels: tuple
    repeats: tuple
    expand_ratio: float
    classifier_hidden: int
    expected_params: int
    checkpoint_prefix: str


WASTENET_256K = WasteNetVariant(
    variant_id="wastenet_256k",
    display_name="WasteNet-256K",
    channels=(16, 32, 72, 128, 224),
    repeats=(1, 1, 2, 1),
    expand_ratio=2.5,
    classifier_hidden=0,
    expected_params=255_777,
    checkpoint_prefix="wastenet_256k",
)


class ConvBNAct(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, groups: int = 1):
        super().__init__(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=kernel_size // 2,
                groups=groups,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )


class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, expand_ratio: float = 1.0):
        super().__init__()
        hidden_channels = int(round(in_channels * expand_ratio))
        layers = []
        if hidden_channels != in_channels:
            layers.append(ConvBNAct(in_channels, hidden_channels, kernel_size=1, stride=1))
        layers.extend([
            ConvBNAct(hidden_channels, hidden_channels, kernel_size=3, stride=stride, groups=hidden_channels),
            nn.Conv2d(hidden_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        ])
        self.block = nn.Sequential(*layers)
        self.activation = nn.SiLU(inplace=True)
        self.use_residual = stride == 1 and in_channels == out_channels

    def forward(self, x):
        y = self.block(x)
        if self.use_residual:
            y = y + x
        return self.activation(y)


class WasteNet(nn.Module):
    def __init__(self, variant: WasteNetVariant, num_classes: int = 6, dropout: float = 0.2):
        super().__init__()
        layers = [ConvBNAct(3, variant.channels[0], kernel_size=3, stride=2)]
        in_channels = variant.channels[0]

        for out_channels, repeat in zip(variant.channels[1:], variant.repeats):
            for block_idx in range(repeat):
                stride = 2 if block_idx == 0 else 1
                layers.append(
                    DepthwiseSeparableBlock(
                        in_channels,
                        out_channels,
                        stride=stride,
                        expand_ratio=variant.expand_ratio,
                    )
                )
                in_channels = out_channels

        self.features = nn.Sequential(*layers)
        if variant.classifier_hidden > 0:
            self.classifier = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Dropout(dropout),
                nn.Linear(in_channels, variant.classifier_hidden),
                nn.SiLU(inplace=True),
                nn.Dropout(dropout),
                nn.Linear(variant.classifier_hidden, num_classes),
            )
        else:
            self.classifier = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Dropout(dropout),
                nn.Linear(in_channels, num_classes),
            )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def create_wastenet_256k() -> WasteNet:
    return WasteNet(variant=WASTENET_256K, num_classes=NUM_CLASSES)


model = create_wastenet_256k().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

sample = torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE).to(device)
with torch.no_grad():
    sample_output = model(sample)
assert tuple(sample_output.shape) == (1, NUM_CLASSES), f"Unexpected output shape: {sample_output.shape}"
assert total_params == WASTENET_256K.expected_params, f"Expected 255,777 params, got {total_params:,}"

print(f"Model        : {cfg.MODEL_NAME}")
print(f"Variant      : {WASTENET_256K.variant_id}")
print(f"Total params : {total_params:,}")
print(f"Trainable    : {trainable_params:,}")
print(f"Output shape : {list(sample_output.shape)}")

del sample, sample_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 10. Teacher, KD Loss, Optimizer, Scheduler
# ============================================================

def create_teacher_model(pretrained: bool = False):
    return timm.create_model(cfg.TEACHER_TIMM_NAME, pretrained=pretrained, num_classes=NUM_CLASSES)


def load_teacher_model(checkpoint_path: Path):
    loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    teacher = create_teacher_model(pretrained=False)
    teacher.load_state_dict(loaded["model_state_dict"])
    teacher = teacher.to(device).eval()
    for parameter in teacher.parameters():
        parameter.requires_grad = False
    print(f"Teacher loaded: {checkpoint_path}")
    print(f"Teacher best epoch: {loaded.get('best_epoch', 'n/a')}")
    print(f"Teacher best val acc: {loaded.get('best_val_acc', 'n/a')}")
    return teacher, loaded


teacher_model, teacher_checkpoint = load_teacher_model(TEACHER_CHECKPOINT_PATH)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=cfg.LR, momentum=cfg.MOMENTUM, weight_decay=cfg.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
scaler = GradScaler(enabled=AMP_ENABLED)


def kd_loss(student_logits, teacher_logits, labels):
    temperature = cfg.KD_TEMPERATURE
    alpha = cfg.KD_ALPHA
    student_log_soft = F.log_softmax(student_logits.float() / temperature, dim=1)
    teacher_soft = F.softmax(teacher_logits.float() / temperature, dim=1)
    loss_soft = F.kl_div(student_log_soft, teacher_soft, reduction="batchmean") * (temperature ** 2)
    loss_hard = criterion(student_logits.float(), labels)
    loss = alpha * loss_soft + (1.0 - alpha) * loss_hard
    return loss, loss_soft, loss_hard

print("Training components ready")


In [ ]:
# ============================================================
# 11. Train / Validate Helpers
# ============================================================

def run_one_train_epoch(student, teacher, loader):
    student.train()
    teacher.eval()
    total_loss = 0.0
    total_soft_loss = 0.0
    total_hard_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=AMP_ENABLED):
            student_logits = student(images)
            with torch.no_grad():
                teacher_logits = teacher(images)
        with autocast(enabled=False):
            loss, loss_soft, loss_hard = kd_loss(student_logits, teacher_logits, labels)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite KD loss detected")
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_soft_loss += loss_soft.item() * batch_size
        total_hard_loss += loss_hard.item() * batch_size
        total_correct += (student_logits.argmax(dim=1) == labels).sum().item()
        total += batch_size
    return total_loss / total, total_correct / total, total_soft_loss / total, total_hard_loss / total


@torch.no_grad()
def run_eval_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        with autocast(enabled=AMP_ENABLED):
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, total_correct / total


In [ ]:
# ============================================================
# 13. Evaluation Helpers
# ============================================================

@torch.no_grad()
def collect_predictions(model, loader, split_name: str) -> pd.DataFrame:
    model.eval()
    rows = []
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].cpu().numpy()
        with autocast(enabled=AMP_ENABLED):
            logits = model(images)
            probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        for i in range(len(labels)):
            row = {
                "sample_id": batch["sample_id"][i],
                "image_path": batch["relative_path"][i],
                "label": CLASS_NAMES[int(labels[i])],
                "label_id": int(labels[i]),
                "prediction": CLASS_NAMES[int(preds[i])],
                "prediction_id": int(preds[i]),
                "seed": cfg.SEED,
                "model_id": cfg.EXPERIMENT_ID,
                "variant": cfg.TRAINING_MODE,
                "split": split_name,
            }
            for class_idx, class_name in enumerate(CLASS_NAMES):
                row[f"prob_{class_name}"] = float(probs[i, class_idx])
            rows.append(row)
    return pd.DataFrame(rows)


def compute_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label_id"].to_numpy()
    y_pred = pred_df["prediction_id"].to_numpy()
    prob_cols = [f"prob_{class_name}" for class_name in CLASS_NAMES]
    y_prob = pred_df[prob_cols].to_numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    try:
        auc_macro_ovr = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except ValueError:
        auc_macro_ovr = np.nan

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "auc_macro_ovr": float(auc_macro_ovr),
    }


def save_confusion_matrix(pred_df: pd.DataFrame, split_name: str):
    cm = confusion_matrix(pred_df["label_id"], pred_df["prediction_id"], labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{cfg.EXPERIMENT_ID} WasteNet-256K Direct KD - {split_name}")
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            ax.text(c, r, str(cm[r, c]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    path = cfg.OUTPUT_DIR / f"confusion_matrix_{split_name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


## 4. Train / Evaluate Driver

In [ ]:
# ============================================================
# 12-17. KD train / eval / save driver (one config)
# ============================================================

def run_one_kd_config(temperature, alpha, setup_id, output_dir, preset=None):
    """Train + evaluate + save WasteNet-256K for one (T, alpha). Returns a val summary dict."""
    global optimizer, scaler

    cfg.KD_TEMPERATURE = temperature
    cfg.KD_ALPHA = alpha
    cfg.SETUP_ID = setup_id
    cfg.OUTPUT_DIR = Path(output_dir)
    cfg.PILOT_PRESET = preset
    cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Reproducible fresh student + identical data order so configs differ only by (T, alpha).
    seed_everything(cfg.SEED)
    generator.manual_seed(cfg.SEED)

    model = create_wastenet_256k().to(device)
    optimizer = optim.SGD(model.parameters(), lr=cfg.LR, momentum=cfg.MOMENTUM, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
    scaler = GradScaler(enabled=AMP_ENABLED)

    history = []
    best_val_acc = -1.0
    best_epoch = 0
    best_model_state = None
    best_monitor_value = np.inf if cfg.EARLY_STOPPING_MONITOR == "val_loss" else -np.inf
    epochs_without_improvement = 0
    total_start = time.time()

    print(f"Starting {cfg.EXPERIMENT_NAME} | seed={cfg.SEED} | T={cfg.KD_TEMPERATURE} | alpha={cfg.KD_ALPHA}")
    for epoch in range(1, cfg.EPOCHS + 1):
        epoch_start = time.time()
        current_lr = optimizer.param_groups[0]["lr"]
        train_loss, train_acc, train_soft_loss, train_hard_loss = run_one_train_epoch(model, teacher_model, train_loader)
        val_loss, val_acc = run_eval_epoch(model, val_loader)
        scheduler.step()
        epoch_time = time.time() - epoch_start
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc
            best_epoch = epoch
            best_model_state = copy.deepcopy(model.state_dict())
        if cfg.EARLY_STOPPING_MONITOR == "val_loss":
            monitor_value = val_loss
            monitor_improved = monitor_value < best_monitor_value - 1e-8
        elif cfg.EARLY_STOPPING_MONITOR == "val_acc":
            monitor_value = val_acc
            monitor_improved = monitor_value > best_monitor_value + 1e-8
        else:
            raise ValueError(f"Unsupported EARLY_STOPPING_MONITOR: {cfg.EARLY_STOPPING_MONITOR}")
        if monitor_improved:
            best_monitor_value = monitor_value
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_loss_soft": train_soft_loss,
            "train_loss_hard": train_hard_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": current_lr,
            "epoch_time_sec": epoch_time,
            "is_best": is_best,
            "early_stop_monitor": cfg.EARLY_STOPPING_MONITOR,
            "early_stop_value": monitor_value,
            "early_stop_improved": monitor_improved,
            "epochs_without_improvement": epochs_without_improvement,
        })
        marker = " BEST" if is_best else ""
        early_stop_status = f" | es_wait={epochs_without_improvement}/{cfg.PATIENCE}" if cfg.EARLY_STOPPING else ""
        print(
            f"Epoch {epoch:03d}/{cfg.EPOCHS} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"soft={train_soft_loss:.4f} hard={train_hard_loss:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
            f"lr={current_lr:.6f} time={epoch_time:.1f}s{early_stop_status}{marker}"
        )
        if cfg.EARLY_STOPPING and epochs_without_improvement >= cfg.PATIENCE:
            print(f"Early stopping triggered at epoch {epoch} ({cfg.EARLY_STOPPING_MONITOR} did not improve for {cfg.PATIENCE} epochs).")
            break

    total_train_time = time.time() - total_start
    history_df = pd.DataFrame(history)
    if best_model_state is None:
        raise RuntimeError("No best model state was captured.")
    print("Training complete")
    print(f"Best epoch: {best_epoch}")
    print(f"Best val acc: {best_val_acc:.6f}")
    print(f"Total minutes: {total_train_time / 60:.1f}")

    model.load_state_dict(best_model_state)

    val_predictions_df = collect_predictions(model, val_loader, "val")
    val_metrics = compute_metrics(val_predictions_df)

    test_predictions_df = pd.DataFrame()
    test_metrics = None
    if cfg.EVALUATE_TEST and cfg.RUN_PHASE == "final":
        test_predictions_df = collect_predictions(model, test_loader, "test")
        test_metrics = compute_metrics(test_predictions_df)
    else:
        print("Independent test evaluation skipped. Set RUN_PHASE='final' and EVALUATE_TEST=True to enable it.")

    print("Validation metrics:")
    display(pd.DataFrame([val_metrics]))
    if test_metrics is not None:
        print("Test metrics:")
        display(pd.DataFrame([test_metrics]))

    val_pred_path = cfg.OUTPUT_DIR / f"predictions_{cfg.EXPERIMENT_ID.lower()}_{cfg.TRAINING_MODE}_{cfg.RUN_PHASE}_seed_{cfg.SEED}_val.csv"
    val_predictions_df.to_csv(val_pred_path, index=False)
    print(f"Saved: {val_pred_path}")

    test_pred_path = None
    if not test_predictions_df.empty:
        test_pred_path = cfg.OUTPUT_DIR / f"predictions_{cfg.EXPERIMENT_ID.lower()}_{cfg.TRAINING_MODE}_{cfg.RUN_PHASE}_seed_{cfg.SEED}_test.csv"
        test_predictions_df.to_csv(test_pred_path, index=False)
        print(f"Saved: {test_pred_path}")

    val_cm_path = save_confusion_matrix(val_predictions_df, "val")
    test_cm_path = None
    if not test_predictions_df.empty:
        test_cm_path = save_confusion_matrix(test_predictions_df, "test")

    prefix = f"{cfg.EXPERIMENT_ID.lower()}_{cfg.TRAINING_MODE}_{cfg.RUN_PHASE}_seed_{cfg.SEED}"
    history_path = cfg.OUTPUT_DIR / f"training_history_{prefix}.csv"
    history_df.to_csv(history_path, index=False)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val")
    axes[0].axvline(best_epoch, color="red", linestyle="--", alpha=0.6)
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[1].plot(history_df["epoch"], history_df["train_acc"], label="train")
    axes[1].plot(history_df["epoch"], history_df["val_acc"], label="val")
    axes[1].axvline(best_epoch, color="red", linestyle="--", alpha=0.6)
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    axes[2].plot(history_df["epoch"], history_df["lr"], color="green", label="lr")
    axes[2].set_title("Learning Rate")
    axes[2].set_xlabel("Epoch")
    axes[2].grid(alpha=0.3)
    fig.suptitle(f"{cfg.EXPERIMENT_ID} WasteNet-256K Direct KD - seed {cfg.SEED} | T={cfg.KD_TEMPERATURE}, alpha={cfg.KD_ALPHA}")
    fig.tight_layout()
    curve_path = cfg.OUTPUT_DIR / f"training_curves_{prefix}.png"
    fig.savefig(curve_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {history_path}")
    print(f"Saved: {curve_path}")

    metrics_rows = [{"split": "val", **val_metrics}]
    if test_metrics is not None:
        metrics_rows.append({"split": "test", **test_metrics})
    metrics_df = pd.DataFrame(metrics_rows)
    metrics_path = cfg.OUTPUT_DIR / f"metrics_{prefix}.csv"
    metrics_df.to_csv(metrics_path, index=False)

    config_dict = {
        "experiment_id": cfg.EXPERIMENT_ID,
        "experiment_name": cfg.EXPERIMENT_NAME,
        "dataset_name": cfg.DATASET_NAME,
        "dataset_version": cfg.DATASET_VERSION,
        "protocol": "R0-K5 stratified 70/15/15",
        "seed": cfg.SEED,
        "run_phase": cfg.RUN_PHASE,
        "setup_id": cfg.SETUP_ID,
        "pilot_preset": cfg.PILOT_PRESET if cfg.RUN_PHASE == "pilot" else None,
        "output_root": str(cfg.OUTPUT_ROOT),
        "output_dir": str(cfg.OUTPUT_DIR),
        "dataset_dir": str(cfg.DATASET_DIR),
        "r0_dir": str(R0_DIR),
        "manifest_path": str(MANIFEST_PATH),
        "teacher_checkpoint_path": str(TEACHER_CHECKPOINT_PATH),
        "teacher_model_name": cfg.TEACHER_MODEL_NAME,
        "teacher_timm_name": cfg.TEACHER_TIMM_NAME,
        "model_name": cfg.MODEL_NAME,
        "variant_id": WASTENET_256K.variant_id,
        "variant": {
            "channels": WASTENET_256K.channels,
            "repeats": WASTENET_256K.repeats,
            "expand_ratio": WASTENET_256K.expand_ratio,
            "classifier_hidden": WASTENET_256K.classifier_hidden,
            "expected_params": WASTENET_256K.expected_params,
        },
        "training_mode": cfg.TRAINING_MODE,
        "knowledge_distillation": True,
        "kd_type": "logits_direct",
        "kd_temperature": cfg.KD_TEMPERATURE,
        "kd_alpha": cfg.KD_ALPHA,
        "pilot_grid_temperatures": cfg.PILOT_GRID_TEMPERATURES,
        "pilot_grid_alphas": cfg.PILOT_GRID_ALPHAS,
        "pilot_anchor": cfg.PILOT_ANCHOR,
        "pilot_configs": cfg.PILOT_CONFIGS,
        "pilot_sweep": cfg.PILOT_SWEEP,
        "img_size": cfg.IMG_SIZE,
        "epochs": cfg.EPOCHS,
        "final_epochs": cfg.FINAL_EPOCHS,
        "pilot_epochs": cfg.PILOT_EPOCHS,
        "batch_size": cfg.BATCH_SIZE,
        "optimizer": "SGD",
        "lr": cfg.LR,
        "momentum": cfg.MOMENTUM,
        "weight_decay": cfg.WEIGHT_DECAY,
        "scheduler": cfg.SCHEDULER,
        "use_amp": AMP_ENABLED,
        "early_stopping": cfg.EARLY_STOPPING,
        "evaluate_test": cfg.EVALUATE_TEST,
        "early_stopping_monitor": cfg.EARLY_STOPPING_MONITOR,
        "patience": cfg.PATIENCE,
        "checkpoint_metric": cfg.CHECKPOINT_METRIC,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "train_size": int(len(train_df)),
        "val_size": int(len(val_df)),
        "test_size": int(len(test_df)),
        "total_params": int(total_params),
        "trainable_params": int(trainable_params),
        "total_train_time_sec": float(total_train_time),
    }
    checkpoint = {
        "model_state_dict": best_model_state,
        "teacher_checkpoint_path": str(TEACHER_CHECKPOINT_PATH),
        "best_epoch": best_epoch,
        "best_val_acc": float(best_val_acc),
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "config": config_dict,
        "class_mapping": class_mapping,
    }
    checkpoint_path = cfg.OUTPUT_DIR / f"wastenet_256k_direct_kd_{prefix}_best.pth"
    torch.save(checkpoint, checkpoint_path)
    config_path = cfg.OUTPUT_DIR / f"config_{prefix}.json"
    config_path.write_text(json.dumps(config_dict, indent=2), encoding="utf-8")
    artifact_manifest = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_id": cfg.EXPERIMENT_ID,
        "dataset_name": cfg.DATASET_NAME,
        "dataset_version": cfg.DATASET_VERSION,
        "seed": cfg.SEED,
        "run_phase": cfg.RUN_PHASE,
        "training_mode": cfg.TRAINING_MODE,
        "pilot_preset": cfg.PILOT_PRESET if cfg.RUN_PHASE == "pilot" else None,
        "kd_temperature": cfg.KD_TEMPERATURE,
        "kd_alpha": cfg.KD_ALPHA,
        "setup_id": cfg.SETUP_ID,
        "output_dir": str(cfg.OUTPUT_DIR),
        "artifacts": {
            "checkpoint": str(checkpoint_path),
            "config": str(config_path),
            "history": str(history_path),
            "metrics": str(metrics_path),
            "val_predictions": str(val_pred_path),
            "test_predictions": str(test_pred_path) if test_pred_path else None,
            "val_confusion_matrix": str(val_cm_path),
            "test_confusion_matrix": str(test_cm_path) if test_cm_path else None,
            "training_curves": str(curve_path),
        },
    }
    artifact_manifest_path = cfg.OUTPUT_DIR / f"artifact_manifest_{prefix}.json"
    artifact_manifest_path.write_text(json.dumps(artifact_manifest, indent=2), encoding="utf-8")
    print(f"Saved checkpoint: {checkpoint_path}")
    print(f"Saved metrics   : {metrics_path}")
    print(f"Saved manifest  : {artifact_manifest_path}")
    display(metrics_df)

    loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    verify_model = create_wastenet_256k()
    verify_model.load_state_dict(loaded["model_state_dict"])

    print("Checkpoint verification passed")
    print(f"Best epoch   : {loaded['best_epoch']}")
    print(f"Best val acc : {loaded['best_val_acc']:.6f}")

    del loaded, verify_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "preset": preset,
        "temperature": temperature,
        "alpha": alpha,
        "best_epoch": best_epoch,
        "best_val_acc": float(best_val_acc),
        "val_accuracy": float(val_metrics["accuracy"]),
        "val_f1_macro": float(val_metrics["f1_macro"]),
        "val_auc": float(val_metrics["auc_macro_ovr"]),
        "test_accuracy": float(test_metrics["accuracy"]) if test_metrics else None,
        "setup_id": setup_id,
        "output_dir": str(cfg.OUTPUT_DIR),
        "checkpoint": str(checkpoint_path),
    }

print("KD config driver ready")

In [ ]:
# ============================================================
# 18. Run: pilot 3-config sweep (one Run All) or single final
# ============================================================

if cfg.RUN_PHASE == "pilot":
    pilot_root = cfg.OUTPUT_ROOT / "pilots" / cfg.EXPERIMENT_ID
    sweep_results = []
    for preset in cfg.PILOT_SWEEP:
        params = cfg.PILOT_CONFIGS[preset]
        T, alpha = params["temperature"], params["alpha"]
        setup_id = build_pilot_setup_id(preset, T, alpha)
        out_dir = pilot_root / setup_id
        print("\n" + "=" * 78)
        print(f"PILOT CONFIG {len(sweep_results) + 1}/{len(cfg.PILOT_SWEEP)}: {preset} | T={T} alpha={alpha}")
        print("=" * 78)
        sweep_results.append(run_one_kd_config(T, alpha, setup_id, out_dir, preset=preset))

    sweep_df = pd.DataFrame(sweep_results)[
        ["preset", "temperature", "alpha", "best_epoch", "val_accuracy", "val_f1_macro", "val_auc", "setup_id"]
    ].sort_values("val_accuracy", ascending=False).reset_index(drop=True)
    pilot_root.mkdir(parents=True, exist_ok=True)
    summary_path = pilot_root / "r7_k5_pilot_sweep_summary.csv"
    sweep_df.to_csv(summary_path, index=False)

    print("\n" + "=" * 78)
    print("PILOT SWEEP SUMMARY (sorted by val accuracy):")
    display(sweep_df)
    winner = sweep_df.iloc[0]
    print(f"Saved summary: {summary_path}")
    print(
        f"Winner on val: {winner['preset']} (T={winner['temperature']}, alpha={winner['alpha']}) "
        f"-> val_acc={winner['val_accuracy']:.4f}, val_f1={winner['val_f1_macro']:.4f}"
    )
    print("Next: set FINAL_KD_TEMPERATURE / FINAL_KD_ALPHA to this winner, RUN_PHASE='final', run 5 seeds.")
else:
    print("\n" + "=" * 78)
    print(f"FINAL RUN | seed={cfg.SEED} | T={cfg.KD_TEMPERATURE} alpha={cfg.KD_ALPHA}")
    print("=" * 78)
    result = run_one_kd_config(cfg.KD_TEMPERATURE, cfg.KD_ALPHA, cfg.SETUP_ID, cfg.OUTPUT_DIR, preset=None)
    print(
        f"\nR7-K5 final complete | seed={cfg.SEED} | best_epoch={result['best_epoch']} | "
        f"val_acc={result['val_accuracy']:.4f} | test_acc={result['test_accuracy']}"
    )

## Output Artifacts

Pilot: tiap config menulis checkpoint + history + metrics + prediction + confusion matrix + training curves + config JSON + artifact manifest ke `pilots/R7/<setup_id>/`, plus satu `r7_k5_pilot_sweep_summary.csv` ringkasan val. Final: artefak lengkap per seed di `final/R7/seed_<seed>/`, dengan independent test set dievaluasi setelah winner dibekukan.